## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Campus device ownership: Wilson score interval

**Confidence Interval Estimation for the Population Proportion** · a worked example from the StatNex teaching guide

**The story.** A campus electronics store surveyed students about which devices they own, asking different numbers of students about each device: 300 about laptops, 500 about tablets and 200 about smartwatches.

**The question.** What share of all students owns each device, and how precise is each estimate?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `02_one_sample/one_sample_proportion_ci.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Variable,Laptop,Tablet,Smartwatch
Sample size,300.0,500.0,200.0
Ownership rate,0.92,0.47,0.21
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)

## The data

In [ ]:
# Each column holds the sample size and the proportion, top to bottom.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Laptop", "Tablet", "Smartwatch"]
confidence = 95 / 100
alpha = 1 - confidence
z = stats.norm.ppf(0.5 + confidence / 2)

## Confidence interval for the proportion: Wilson score interval

In [ ]:
# Every method gives the range of population shares these answers are consistent with.
# The interval covers only the luck of who was asked: it assumes a random sample of
# independent answers, and says nothing about a biased sample or non-response.
results = {}
for name in variables:
    size, p = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:2]
    n = int(round(size))
    x = n * p  # the number of "yes" answers
    # The Wilson score interval: the p0 values a score test would not reject. Its
    # coverage is close to the stated level even for small n.
    lower, upper = proportion_confint(x, n, alpha, method="wilson")
    # A proportion lies between 0 and 1, so the limits are cut there.
    results[name] = {"n": n, "lower": max(0.0, lower), "proportion": p,
                     "upper": min(1.0, upper)}

table = pd.DataFrame(results)
print(table)
print(f"\nConfidence level: {confidence:.1%}")

## What the result says

In the survey, 92% of the 300 students asked own a laptop, 47% of 500 own a tablet and 21% of 200 own a smartwatch. A confidence interval is a range of values the data agree with for the true share among all students.

This run builds it with the Wilson score interval, a method that stays reliable for small samples and for shares near 0% or 100%, and that gives slightly narrower ranges than the cautious Clopper-Pearson exact method. At the 95% confidence level, the laptop interval runs from 88.4% to 94.6%, the tablet interval from 42.7% to 51.4%, and the smartwatch interval from 15.9% to 27.2%.

The choice of method changes little here. The smartwatch estimate is still the least precise, because only 200 students were asked about it.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.